In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q05/q05_rewrite/checkpoints/pre_cell_6.pickle

In [4]:
%%RecordEvent
%%time
### cell 6 ###
# Filter on region and order year using GPU-friendly dt.year to avoid CPU-side Timestamp comparisons
rsel = region.R_NAME == "ASIA"
fregion = region[rsel]

# Use dt.year for a single GPU comparison instead of two CPU comparisons
osel = orders.O_ORDERDATE.dt.year == 1996
forders = orders[osel]

# Perform the sequence of merges as before (all GPU)
jn1 = fregion.merge(nation,   left_on="R_REGIONKEY", right_on="N_REGIONKEY")
jn2 = jn1.merge( customer,    left_on="N_NATIONKEY", right_on="C_NATIONKEY")
jn3 = jn2.merge( forders,     left_on="C_CUSTKEY",   right_on="O_CUSTKEY")
jn4 = jn3.merge( lineitem,    left_on="O_ORDERKEY",  right_on="L_ORDERKEY")
jn5 = supplier.merge(
    jn4,
    left_on=["S_SUPPKEY", "S_NATIONKEY"],
    right_on=["L_SUPPKEY", "N_NATIONKEY"]
)

# Compute revenue and aggregate
jn5["TMP"] = jn5.L_EXTENDEDPRICE * (1.0 - jn5.L_DISCOUNT)
total = (
    jn5.groupby("N_NAME", as_index=False, sort=False)["TMP"]
       .sum()
       .sort_values("TMP", ascending=False)
)

CPU times: user 113 ms, sys: 36.3 ms, total: 149 ms
Wall time: 156 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q05/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_6_try_0.pickle